# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema for this dataset defines tabular record sets. We'll enumerate their `@id` values and fields. If there are no record sets directly listed in the metadata, we'll discover them from the dataset object.

In [ ]:
# Discover all record sets and their fields
record_sets = dataset.record_sets
# record_sets is a list of mlcroissant.RecordSet objects

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']}, Name: {field.get('name', '')}, DataType: {field.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
# Collect the @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show sample columns and records for the first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns for RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We'll demonstrate removing records below a numeric threshold, normalizing, and grouping by a categorical field, using available columns (referenced by their `@id`).

In [ ]:
# Identify numeric and group fields
df = dataframes[main_record_set_id]
# Attempt to select a numeric field (e.g., Age or diagnosis interval)
numeric_field_id = None

# Try 'age' or similar field
possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype in ['int64','float64']]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]

# Try grouping by MSI/MMR status or anatomical location
possible_group_fields = [col for col in df.columns if 'msi' in col.lower() or 'anatomical' in col.lower() or 'sex' in col.lower()]
group_field_id = possible_group_fields[0] if possible_group_fields else None

if numeric_field_id:
    threshold = 50  # Example threshold for age
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot grouped by group_field_id
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 colorectal cancer survivors dataset using `mlcroissant`, referencing all entities by their `@id`.

We examined available record sets and fields, performed basic EDA (filtering, normalization, grouping), and visualized numeric distributions. Further clinical or statistical analysis can be performed based on the dataset field definitions and use cases described in the metadata.

For more information and reproducibility, see [FAIR^2 dataset documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p).